# Noether Framework — Aero CFD Guide

This notebook is a self-contained guide to the [Noether Framework](https://noether-docs.emmi.ai/noether)
for aerodynamic CFD. It covers:

1. [What is Noether](#1-what-is-noether) — framework overview, design principles, and model/dataset zoos
2. [Key concepts](#2-key-concepts) — configuration system, data pipeline, factory pattern
3. [Model architectures: UPT vs AB-UPT](#3-model-architectures-upt-vs-ab-upt) — detailed comparison
4. [Tutorial walkthrough](#4-tutorial-walkthrough) — project structure and component-by-component guide
5. [Practical guide to the source code](#5-practical-guide-to-the-source-code) — factory, training loop, callbacks, optimizer, distributed, extension points
6. [Hands-on: training via CLI](#5-hands-on-training-via-cli) — run experiments with `noether-train`
7. [Hands-on: training via Python preset API](#6-hands-on-training-via-python-preset-api) — programmatic configs
8. [Hands-on: exploring components](#7-hands-on-exploring-components) — load datasets, models, configs interactively
9. [Running at scale on Slurm](#8-running-at-scale-on-slurm) — batch jobs and multi-GPU
10. [Reference from docs](#9-reference-from-docs) — data pipeline internals, design principles, performance tips


**Interactive session setup** (run before launching Jupyter):
```bash
salloc --cpus-per-task=28 --mem=250GB --reservation=dev --gpus-per-node=1 --time 1-0 srun --pty zsh
cd ~/exp/noether && source .venv/bin/activate
jupyter notebook --no-browser --port=8888
```

## 1. What is Noether

[Noether](https://github.com/Emmi-AI/noether) is Emmi AI's open framework for **Engineering AI**.
Built on transformer building blocks optimized for physical systems, it provides the full stack
for building, training, and operating industrial simulation models — targeting domains like
aerodynamic CFD where data lives on irregular meshes and point clouds rather than regular grids.

### Design principles

The framework rests on five architectural pillars
(see [design principles](https://noether-docs.emmi.ai/noether/design_principles_and_limitations.html)):

1. **Configuration-Driven Development (CDD)** — behaviour is separated from implementation
   through dedicated configuration objects (Pydantic models). Components receive typed configs,
   not raw arguments.
2. **Factory pattern & dynamic instantiation** — classes are resolved at runtime via string-based
   class paths (`kind` field), decoupling config from imports.
3. **Strict type safety & runtime validation** — Python type hints + Pydantic validators catch
   errors before training begins. `extra="forbid"` on schemas means typos in YAML are caught immediately.
4. **Defensive programming** — explicit input validation and state guards prevent silent failures.
5. **Composition over inheritance** — complex behaviours emerge through object wrapping
   (e.g. dataset wrappers) rather than deep class hierarchies.

**Two ways to work:**
- **Config-driven** (recommended starting point) — YAML files define everything; use `noether-train` CLI
- **Code-driven** — Python preset API builds configs programmatically; call `HydraRunner` directly

### Model zoo

| Model | Paper | Type | Implementation |
|-------|-------|------|----------------|
| **AB-UPT** | [arXiv:2502.09692](https://arxiv.org/abs/2502.09692) | Full-stack (encoder + decoder included) | `noether.modeling.models.ab_upt` |
| **UPT** | [arXiv:2402.12365](https://arxiv.org/abs/2402.12365) | Full-stack | `noether.modeling.models.upt` |
| **Transformer** | — | Backbone (needs embedding/projection wrapper) | `noether.modeling.models.transformer` |
| **Transolver** | [arXiv:2402.02366](https://arxiv.org/abs/2402.02366) | Backbone (Physics-Attention) | `noether.modeling.models.transolver` |
| **Transolver++** | [arXiv:2502.02414](https://arxiv.org/abs/2502.02414) | Schema-based variant | Config-driven |

**Backbone** models need a wrapper to handle input embeddings and output projections.
**Full-stack** models include the complete pipeline with integrated preprocessing and output layers.

### Dataset zoo

All datasets are in `noether.data.datasets.cfd` and share the `AeroDataset` interface:

| Dataset | Class | Source |
|---------|-------|--------|
| **ShapeNet-Car** | `ShapeNetCarDataset` | [ShapeNet](https://shapenet.org/) |
| **AhmedML** | `AhmedMLDataset` | [CAEML](https://caeml.org/) |
| **DrivAerML** | `DrivAerMLDataset` | [CAEML](https://caeml.org/) |
| **DrivAerNet++** | `DrivAerNetDataset` | [DrivAerNet](https://github.com/Mohamedelrefaie/DrivAerNet) |
| **Emmi Wing** | `EmmiWingDataset` | Internal |

## 2. Key concepts

### Configuration system

Noether uses [**Hydra**](https://hydra.cc/) for hierarchical YAML composition and
[**Pydantic**](https://docs.pydantic.dev/) for runtime type validation.

The configuration flows in three layers:

1. **Base configs** — one YAML per component (dataset, model, trainer, pipeline, optimizer, ...)
2. **Experiment configs** — compose and override base configs.  
   E.g. `+experiment/shapenet=transformer` selects the Transformer model, Lion optimizer, float16 precision
3. **CLI overrides** — quick sweeps without touching files:  
   `trainer.max_epochs=100 model.hidden_dim=256`

The top-level entry point (e.g. `train_shapenet.yaml`) is the **composition root**. It references
sub-configs via Hydra's `defaults` list. Variables like `${dataset_root}` are defined once and
interpolated everywhere. Fields marked `???` are required and must be filled by experiment configs.

Every instantiable class has a **`kind`** field — a Python class path (e.g. `tutorial.model.Transformer`)
used by the **Factory pattern** to dynamically import and instantiate the object with its validated config.

### Data pipeline

Data flows from disk to training batches in four stages, orchestrated by the
[`MultiStagePipeline`](https://noether-docs.emmi.ai/noether/understanding_the_data_pipeline.html):

```
Stage 1: Dataset loading
  .pt files on disk → getitem_* methods → per-tensor normalization (@with_normalizers)
  
Stage 2: Sample processing (per-sample, before batching)
  → Create default tensors (e.g. surface SDF = 0)
  → Subsample mesh points to fixed counts (surface, volume, query/anchor)
  → Rename fields to 'targets' based on model type
  
Stage 3: Collation (samples → batch)
  → DefaultCollator (concatenation) or SparseTensorOffsetCollator (for supernodes)
  
Stage 4: Batch processing (post-collation)
  → Optional transforms on the full batch
```

**The `getitem_*` pattern:** instead of a monolithic `__getitem__`, each tensor has its own
loading method (e.g. `getitem_surface_pressure`, `getitem_volume_velocity`). The framework
discovers them via introspection. This enables:

- **Selective loading** — `excluded_properties` in config skips tensors a model doesn't need
- **Per-tensor normalization** — `@with_normalizers("surface_pressure")` applies MeanStd normalization
- **Easy extensibility** — add a new field by adding one method

**Point sampling:** CFD meshes have thousands to millions of points. `PointSamplingSampleProcessor`
subsamples on-the-fly with shared random permutations (preserving position↔field correspondence).
With `seed=None` (training), each epoch sees different random points — providing implicit augmentation.
For evaluation, `RepeatWrapper` tiles the dataset 10x and averages predictions for stability.

**Stochastic coverage example (ShapeNet-Car, 28,504 volume points sampled to 4,096):**
- Per-epoch probability per point: ~14.4%
- Expected coverage after 13 epochs: ~87%
- Expected coverage after 19 epochs: ~95%

### Dataset hierarchy for CFD

```
torch.utils.data.Dataset
  └── noether.data.Dataset          (getitem_* pattern, normalizer support)
        └── AeroDataset             (common CFD aerodynamics API)
              ├── ShapeNetCarDataset
              ├── AhmedMLDataset
              ├── DrivAerMLDataset
              ├── DrivAerNetDataset
              └── EmmiWingDataset
```

Available tensors per sample (ShapeNet-Car):

| Tensor | Shape | Description |
|--------|-------|-------------|
| `surface_position` | (N_surf, 3) | 3D coordinates of surface mesh |
| `surface_pressure` | (N_surf, 1) | Pressure at surface points |
| `surface_normals` | (N_surf, 3) | Normal vectors at surface |
| `volume_position` | (N_vol, 3) | 3D coordinates of volume mesh |
| `volume_velocity` | (N_vol, 3) | Velocity vectors at volume points |
| `volume_normals` | (N_vol, 3) | Normals pointing to nearest surface |
| `volume_sdf` | (N_vol, 1) | Signed distance to nearest surface |

Note: there is no `surface_sdf` — it's always zero (points on the surface). The pipeline
creates this constant tensor automatically when needed.

### Surface mesh vs volume mesh

In a CFD simulation of airflow around a car (or wing), the computational domain has two
distinct regions that capture different physics:

- **Surface mesh** — the triangulated skin of the body (car, wing, etc.). Points lie *on* the
  geometry. Physical quantities here are what the surface "feels":
  - **Pressure** — force per area pushing on the surface (determines drag/lift coefficients)
  - **Friction / wall shear stress** — tangential drag force from the airflow

- **Volume mesh** — points filling the 3D space *around* the body (the air domain). Physical
  quantities describe the flow field:
  - **Velocity** — how fast air moves at each point
  - **Pressure** — static pressure in the flow
  - **Vorticity** — rotation/turbulence in the flow

Both are needed because they capture complementary information:
- Surface fields → **aerodynamic forces** (drag, lift, downforce) — what engineers optimize for
- Volume fields → **flow structure** (wake, separation, recirculation) — explains *why* forces
  look the way they do

The models predict both simultaneously. This is also why the trainer uses a **weighted
multi-field loss** — you can control how much the model focuses on surface accuracy vs
volume accuracy.

### Field availability across datasets

The datasets are **not** all the same. The `AeroDataset` base class defines `getitem_*` methods
for all possible fields, but each concrete dataset only has data for a subset. This is why
`excluded_properties` exists — you skip what doesn't exist to avoid loading errors.

| Field | ShapeNet-Car | AhmedML | DrivAerML | DrivAerNet | EmmiWing |
|-------|:---:|:---:|:---:|:---:|:---:|
| `surface_position` | yes | yes | yes | yes | yes |
| `surface_pressure` | yes | yes | yes | yes | yes |
| `surface_friction` | — | yes | yes | yes | yes |
| `surface_normals` | yes | — | — | — | — |
| `volume_position` | yes | yes | yes | yes | yes |
| `volume_velocity` | yes | yes | yes | yes | yes |
| `volume_pressure` | — | yes | yes | yes | yes |
| `volume_vorticity` | — | yes | yes | yes | yes |
| `volume_friction` | — | — | — | yes | — |
| `volume_sdf` | yes | — | — | — | — |
| `volume_normals` | yes | — | — | — | — |
| `design_parameters` | — | — | — | — | yes |

**Key differences:**

- **ShapeNet-Car** is the simplest — only surface pressure + volume velocity, plus geometric
  features (normals, SDF). No friction, no volume pressure, no vorticity.
- **AhmedML / DrivAerML / EmmiWing** share the richest common field set — full surface
  (pressure + friction) and full volume (pressure + velocity + vorticity).
- **DrivAerNet** is the only one with `volume_friction`.
- **EmmiWing** is the only one with **design parameters** (geometry: chord, span, taper, sweep,
  dihedral; inflow: velocity, angle of attack) — enabling conditional models that predict flow
  as a function of wing shape.

This directly affects the trainer field weights per dataset:
```python
# ShapeNet-Car (2 fields)
field_weights = {"surface_pressure": 1.0, "volume_velocity": 1.0}

# AhmedML / DrivAerML (5 fields)
field_weights = {
    "surface_pressure": 1.0, "surface_friction": 1.0,
    "volume_pressure": 1.0, "volume_velocity": 1.0, "volume_vorticity": 1.0,
}
```

## 3. Model architectures: UPT vs AB-UPT

Both are transformer-based models for predicting physical fields (pressure, velocity, etc.) on
aerodynamic meshes. They share a **SupernodePooling** geometry encoder but differ fundamentally
in how they route information between surface and volume domains.

### Shared component: SupernodePooling

Compresses a point cloud (potentially millions of points) into a manageable set of "supernodes"
via graph-based neighborhood aggregation:

1. **Graph construction** — KNN or radius search from all points to supernode points
2. **Message embedding** — embed relative/absolute positions between neighbors and supernodes;
   optionally embed physics features (SDF, normals, etc.)
3. **Message aggregation** — scatter-reduce (sum/mean/max) neighbor messages to each supernode
4. **Output** — `(B, num_supernodes, hidden_dim)` dense batch tensor

This is O(kN) where k is neighborhood size — efficient even for very large meshes.

### UPT (Universal Physics Transformer)

Simple single-path architecture ([arXiv:2402.12365](https://arxiv.org/abs/2402.12365)):

```
Surface mesh points
    │
    ▼
SupernodePooling (graph aggregation)
    │  → (B, num_supernodes, hidden_dim)
    ▼
Approximator Blocks (self-attention on supernodes, N layers)
    │  → (B, num_supernodes, hidden_dim)
    ▼
Query Embedding (continuous sincos of query positions)
    │  → (B, num_queries, hidden_dim)
    ▼
Perceiver Decoder (queries cross-attend to supernodes, N layers)
    │  → (B, num_queries, hidden_dim)
    ▼
LayerNorm → Linear Projection
    │
    ▼
Predictions (B, num_queries, output_dim)
```

**Key idea:** query points (where you want predictions) cross-attend to supernode representations
in a fixed perceiver decoder. One output head for all fields.

### AB-UPT (Anchored Branched UPT)

Dual-branch architecture with configurable interaction patterns
([arXiv:2502.09692](https://arxiv.org/abs/2502.09692)):

```
Geometry mesh              Surface anchors + queries     Volume anchors + queries
    │                              │                              │
    ▼                              ▼                              ▼
SupernodePooling          Pos embed + bias MLP           Pos embed + bias MLP
    │                              │                              │
    ▼                              └──────── Concatenate ─────────┘
Geometry Blocks                              │
(self-attention)                             ▼
    │                         x_physics (B, total_tokens, hidden_dim)
    │                                        │
    │                              Physics Blocks (configurable sequence):
    └────────────────────→  ├─ "perceiver" → cross-attend to geometry encoding
                            ├─ "self"      → within-branch attention
                            ├─ "cross"     → between-branch attention
                            └─ "joint"     → all-to-all attention
                                             │
                                    ┌────────┴────────┐
                             Surface split       Volume split
                                    │                 │
                                    ▼                 ▼
                           Surface Decoder    Volume Decoder
                           (self-attention)   (self-attention)
                                    │                 │
                                    ▼                 ▼
                           Linear proj.       Linear proj.
                                    │                 │
                                    ▼                 ▼
                           surface_pressure   volume_velocity
                           surface_friction   volume_pressure
                           ...                volume_vorticity ...
```

### Anchor points vs query points

- **Anchors** are fixed sample points on the surface/volume mesh whose key/value representations
  get **cached**. Computed once, reused for arbitrary queries.
- **Queries** are dynamic evaluation points — different each forward pass, cross-attend to
  cached anchors for efficient inference at varying resolutions.

### The `physics_blocks` sequence

This is the main architectural knob in AB-UPT. A list like `["perceiver", "self", "cross"] * 5`
defines the sequence of attention patterns in the physics reasoning stage:

| Block type | What it does |
|------------|-------------|
| `"perceiver"` | Cross-attend to geometry encoding — injects geometric context from supernodes |
| `"self"` | Surface tokens attend to surface anchors, volume to volume (independent refinement) |
| `"cross"` | Surface tokens attend to volume anchors and vice versa (couples the two domains) |
| `"joint"` | All tokens attend to all anchors (full interaction) |

Under the hood, these use a **MixedAttention** module that routes attention via named
`TokenSpec` / `AttentionPattern` objects — flexible, cacheable, and batchable.

### Side-by-side comparison

| Aspect | UPT | AB-UPT |
|--------|-----|--------|
| **Architecture** | Single-path | Dual-branch (surface + volume) |
| **Encoder** | SupernodePooling | SupernodePooling + geometry blocks |
| **Physics processing** | Fixed perceiver decoder | Configurable `physics_blocks` sequence |
| **Output** | Single head for all fields | Separate surface & volume decoders |
| **Points** | Query points only | Anchors (cached) + queries (dynamic) |
| **Surface/volume coupling** | Implicit (shared representation) | Explicit (cross/joint attention) |
| **KV caching** | Limited | Full — cache anchors, reuse for queries |
| **Conditioning** | None | Optional geometry/inflow design parameters |
| **Complexity** | ~125 lines | ~600 lines |

### RoPE (Rotary Position Embedding)

Both models use RoPE to encode spatial positions in attention. Applied to Q and K (not V),
it encodes absolute position as rotations so that relative position bias emerges naturally —
critical for sparse, non-sequential spatial inputs.

## 4. Tutorial walkthrough

The `tutorial/` directory implements a complete Noether project based on Section 4.4 of the
[AB-UPT paper](https://arxiv.org/pdf/2502.09692). It demonstrates every core framework concept
using aerodynamic CFD datasets.

### Project structure

```
tutorial/
├── callbacks/       # Evaluation, logging & monitoring hooks
├── configs/         # Hydra YAML configs for every component
│   ├── experiment/  # Per-dataset model overrides (shapenet/, ahmedml/, drivaerml/, ...)
│   ├── model/       # Model architecture definitions
│   ├── pipeline/    # Multi-stage data pipeline config
│   ├── optimizer/   # Optimizer configs (AdamW, Lion, ...)
│   ├── trainer/     # Trainer loop settings (epochs, precision, loss weights, ...)
│   ├── datasets/    # Dataset split configs (train/test, wrappers)
│   ├── dataset_normalizers/  # Per-tensor normalization (MeanStd, etc.)
│   ├── dataset_statistics/   # Precomputed mean/std per field
│   ├── data_specs/  # Field definitions (names, dims, types per dataset)
│   ├── tracker/     # Experiment tracking (W&B or disabled)
│   ├── slurm/       # Slurm resource config
│   └── train_*.yaml # Top-level entry points per dataset
├── jobs/            # Slurm .job scripts + experiment lists (model x seed arrays)
├── model/           # Model wrappers (Transformer, UPT, AB-UPT, Transolver, composite)
├── pipeline/        # Multi-stage data pipeline implementation
├── schemas/         # Pydantic schemas for config validation of all components
└── trainers/        # Trainer classes (training loop + weighted loss computation)
```

**Minimal required structure** for any Noether project:
```
├── callbacks/   # Can be empty if using only defaults
├── configs/     # Required: defines all configurations
├── model/       # Required: defines model architectures
├── pipeline/    # Required: defines data processing
└── trainers/    # Required: defines training logic
```

### Component-by-component guide

#### Configs

The composition root `train_shapenet.yaml` pulls in sub-configs via Hydra `defaults`:

```yaml
dataset_root: <path to your shapenet dataset root>
dataset_kind: noether.data.datasets.cfd.ShapeNetCarDataset
config_schema_kind: tutorial.schemas.config_schema.TutorialConfigSchema

defaults:
  - data_specs: shapenet_car
  - dataset_normalizers: shapenet_dataset_normalizers
  - model: ???           # filled by experiment config
  - trainer: shapenet_trainer
  - datasets: shapenet_dataset
  - tracker: ???         # filled by experiment config
  - pipeline: shapenet_pipeline
  - optimizer: adamw
  - _self_
```

An experiment config like `experiment/shapenet/transformer.yaml` fills the `???` fields:

```yaml
defaults:
  - override /model: transformer
  - override /tracker: development_tracker
  - override /optimizer: lion
trainer:
  precision: float16
```

#### Dataset

Datasets use the `getitem_*` pattern. Each method loads one tensor:

```python
@with_normalizers("surface_pressure")
def getitem_surface_pressure(self, idx: int) -> torch.Tensor:
    return self._load(idx=idx, filename="surface_pressure.pt").unsqueeze(1)
```

Normalizers are configured in YAML and applied automatically via the decorator.
Use `excluded_properties` to skip tensors a model doesn't need.

To compute statistics for normalizers:
```bash
noether-dataset-stats \
  --dataset_kind=noether.data.datasets.cfd.ShapeNetCarDataset \
  --root=/path/to/data --split=train \
  --exclude_attributes=volume_velocity,surface_normals
```

#### Pipeline

The `AeroMultistagePipeline` builds three processor lists:

1. **Sample processors** — default tensor creation, point subsampling, target renaming
2. **Collators** — `DefaultCollator` for most; `SparseTensorOffsetCollator` for supernodes
3. **Batch processors** — (unused in this tutorial)

The pipeline adapts to the model type:
- **Point-based** (Transformer, Transolver): input points = prediction points
- **Query-based** (UPT, AB-UPT): separate query/anchor points for predictions

#### Models

All models inherit from `noether.core.models.Model`. The tutorial's `BaseModel` adds shared
utilities: positional embeddings, physics feature projection, surface/volume bias, and
`gather_outputs` (splits raw output tensor into named physical quantities).

Schemas use multiple inheritance:
```
ModelBaseConfig (kind, name, optimizer, initializers, frozen, forward_properties)
    └── TransformerConfig (inherits TransformerBlockConfig + ModelBaseConfig)
          └── depth, hidden_dim, num_heads, mlp_expansion_factor, attention_constructor, use_rope, ...
```

For UPT/AB-UPT, top-level parameters like `hidden_dim` automatically propagate to submodules
(supernode pooling, approximator, decoder) unless explicitly overridden.

**Composite models** allow sub-modules with independent optimizers, LR schedules, and freeze
states — optimizers attach to models, not globally.

#### Trainer

The `BaseTrainer` manages the full training loop. You implement:

- **`loss_compute(forward_output, targets)`** — your task-specific loss function
- **`train_step(batch, model)`** (optional override) — default splits batch into
  `forward_batch` → `model(**forward_batch)` and `targets_batch` → `loss_compute`

The tutorial's `AutomotiveAerodynamicsCFDTrainer` uses a **weighted multi-field loss** with two
levels: individual field weights (`surface_pressure_weight`, `volume_velocity_weight`, ...)
and group weights (`surface_weight`, `volume_weight`).

#### Callbacks

Callbacks hook into the training loop. The tutorial uses `PeriodicDataIteratorCallback`:

1. **`process_data(batch)`** — run model inference, compute per-batch metrics
2. **`process_results(results)`** — aggregate and log metrics (e.g. to W&B)

Callbacks access the trainer, model, and data container (for denormalization via
`normalizer.inverse()`). They trigger at intervals via `every_n_epochs`, `every_n_updates`,
or `every_n_samples`.

Computed metrics: **MSE**, **MAE**, **Relative L2 Error**.

Final evaluation uses `test_repeat` (10x `RepeatWrapper`) triggered at `every_n_epochs: ${trainer.max_epochs}`
to reduce variance from stochastic point sampling.

#### Schemas

Every component has a Pydantic schema. The top-level `TutorialConfigSchema` ties them together:

```python
class TutorialConfigSchema(ConfigSchema):
    data_specs: AeroDataSpecs
    model: AnyModelConfig = Field(..., discriminator="name")
    trainer: AutomotiveAerodynamicsCfdTrainerConfig
    datasets: dict[str, AeroDatasetConfig]
    dataset_statistics: AeroStatsSchema | None = None
```

`AnyModelConfig` is a `Union` of all model configs; Pydantic uses the `name` field as discriminator
to select the right schema for validation.

### Scaffolding a new project

The `noether-init` CLI generates a complete, runnable project skeleton with all required
components pre-wired. This is the recommended way to start a new Noether project.

```bash
# Generate a project (options: --tracker wandb/tensorboard/disabled, --hardware gpu/mps/cpu)
uvx --from emmiai-noether noether-init my_project

# Or if noether is already installed:
uv run noether-init my_project

# Train immediately:
cd my_project
uv run noether-train --hp my_project/configs/base_experiment.yaml
```

**What it creates:**

```
my_project/
├── callbacks/
│   └── base.py              # BoilerplateCallback (PeriodicDataIteratorCallback)
├── configs/
│   ├── base_experiment.yaml  # Top-level composition root (model, trainer, datasets, tracker)
│   ├── inference_config.yaml
│   └── tracker/             # W&B, TensorBoard, trackio, or disabled
├── datasets/
│   └── base.py              # Synthetic multi-class classification dataset (generates clusters on a circle)
├── models/
│   └── base.py              # MLP with configurable depth, skip connections, activation, normalization
├── pipelines/
│   ├── collators/           # (empty — uses DefaultCollator)
│   ├── preprocessors/       # (empty — no sample processing needed for synthetic data)
│   └── postprocessors/      # (empty — no batch processing needed)
├── schemas/
│   ├── datasets/            # BaseDatasetConfig (num_samples, num_classes, noise, radius)
│   ├── models/              # BaseModelConfig (input_dim, hidden_dim, output_dim, dropout, ...)
│   └── collator/            # BaseCollatorConfig
├── trainer/
│   └── base.py              # BoilerplateTrainer (overrides train_step with cross-entropy loss)
└── pyproject.toml
```

The generated project is intentionally simple — a synthetic classification task with an MLP —
so you can verify everything works end-to-end before replacing components with your own:

- **Dataset:** generates 2D points in clusters arranged on a circle (configurable classes, noise, radius).
  Uses `getitem_x` and `getitem_y` — the same `getitem_*` pattern as the CFD datasets.
- **Model:** MLP inheriting `noether.core.models.Model` with configurable hidden layers,
  activation (relu/gelu/tanh/sigmoid), normalization (LayerNorm/BatchNorm), dropout, skip connections.
- **Trainer:** overrides `train_step` directly (simpler alternative to implementing `loss_compute`).
  Computes cross-entropy loss.
- **Config:** `base_experiment.yaml` wires everything together — model with AdamW + cosine LR schedule,
  50 epochs, batch size 8, with checkpoint and evaluation callbacks.

The `__PROJECT__` placeholders in templates are replaced with your project name at generation time,
so all `kind` paths resolve correctly (e.g. `my_project.models.BaseModel`).

**To evolve the scaffold into a real project**, replace one component at a time:
swap the dataset for your data, swap the MLP for a transformer, adjust the trainer loss, etc.
The scaffold ensures the config wiring, schema validation, and factory instantiation are already correct.

## 5. Practical guide to the source code

What you need to know when working with `src/noether/` as an ML researcher.

### Package map

```
src/noether/
├── core/          # Engine: factory, training loop, callbacks, optimizer, distributed, checkpointing
├── data/          # Dataset base classes, pipeline, samplers, normalizers, wrappers
├── modeling/      # Models, attention, blocks, layers, encoders, decoders, functional ops
├── training/      # Trainers (BaseTrainer, WeightedLossTrainer), HydraRunner, CLI
├── inference/     # InferenceRunner, CLI
├── io/            # Checkpoint providers (local, S3, HF Hub), disk cache, logging
└── scaffold/      # noether-init project generator
```

### Factory & instantiation

Everything is instantiated via `Factory.create(config)` using the `kind` field as a class path.
The factory handles both Pydantic models and raw dicts. `OptimizerFactory` returns **partials**
(not instantiated objects) because optimizers need model parameters at init time.

Dependencies like `UpdateCounter`, `PathProvider`, etc. are injected as kwargs to constructors —
you don't import them yourself.

### Training loop internals (BaseTrainer)

The core loop in `training/trainers/base.py`:
1. For each epoch, iterate over batches from `InterleavedSampler` (which mixes train + eval)
2. Gradient accumulation via `config.accumulation_steps`
3. Auto-detect precision (fp32/fp16/bfloat16) with `torch.autocast` + `GradScaler`
4. Distributed: auto-wraps model in `DistributedDataParallel`
5. NaN losses are skipped by default
6. `UpdateCounter` tracks all 3 dimensions (epoch, update, sample) simultaneously

**Extension points:**
- Override `loss_compute(forward_output, targets)` — most common
- Override `train_step(batch, model)` — for custom forward logic
- `WeightedLossTrainer` is a generic multi-task trainer: `field_weights={field: weight}`,
  `loss_fn="mse" | "l1" | "huber" | dotted.path.to.custom`

### Optimizer wrapper

The `OptimizerWrapper` in `core/optimizer/` adds features on top of PyTorch optimizers:
- **Param group modifiers** — per-layer LR/WD (e.g. layer-wise LR decay)
- **Gradient clipping** — `clip_grad_value` or `clip_grad_norm`
- **Weight decay exclusion** — auto-excludes biases and norm layer params
- **Dual schedules** — separate LR schedule and WD schedule simultaneously
- Param groups with identical properties are merged for logging efficiency

### Callback system

Three base classes in `core/callbacks/`:

| Base class | Use for | Key methods |
|------------|---------|-------------|
| `CallbackBase` | Non-periodic hooks | `before_training()`, `after_training()` |
| `PeriodicCallback` | Periodic without data | `track_after_update_step()` |
| `PeriodicDataIteratorCallback` | Periodic with dataset iteration | `process_data(batch)`, `process_results(results)` |

**Gotchas:**
- Callbacks can be **stateful** via `state_dict()` / `load_state_dict()` — saved in checkpoints
- Only ONE interval type allowed (`every_n_epochs` XOR `every_n_updates` XOR `every_n_samples`)
- In distributed mode, results are auto-gathered with `all_gather_nograd_clipped`
- Checkpoint keys must be unique across callbacks (raises error on duplicates)

**Built-in callbacks:**
- `CheckpointCallback` — periodic save with size estimation
- `BestCheckpointCallback` — save best by metric, with tolerance for multiple "bests"
- `EmaCallback` — exponential moving average of model parameters
- `OfflineLossCallback` — compute loss on eval set
- Early stoppers — raise `EarlyStopIteration` when criterion met

### Distributed training

In `core/distributed/`:
- Auto-detects SLURM env vars (`SLURM_PROCID`, `SLURM_LOCALID`, etc.)
- `run_managed()` for SLURM, `run_unmanaged()` for local `torch.distributed`
- **Data rank0** is `local_rank0 OR world_size==1` (not always global rank0)
  — matters for multi-node: only one process per node copies data
- `DistributedSampler` requires `drop_last=True` for consistent lengths across ranks

### Interleaved sampling

Instead of separate train/eval DataLoaders, a single `InterleavedSampler` serves both.
It yields training indices continuously, pausing at configured intervals to run complete
evaluation passes. Training stops at the first limit reached: `max_epochs`, `max_updates`,
or `max_samples`.

### Initializers

In `core/initializers/` — for loading pretrained weights or resuming:

| Initializer | Use case |
|-------------|----------|
| `ResumeInitializer` | Resume from previous run's latest checkpoint |
| `PreviousRunInitializer` | Transfer learning from a different run |
| `CheckpointInitializer` | Load from arbitrary checkpoint path |

These restore model weights, optimizer state, callback state, and training iteration.

### Checkpoint system

In `io/checkpoint/`:
- Separate files for weights, optimizer, model config
- **Provider pattern**: abstracts local files, HuggingFace Hub, S3, Azure
- Smart downloading with caching and free-space checks
- Metadata tracks `training_iteration` ("E10_U200_S800" format) for "latest" resolution

### Modeling building blocks

Everything in `modeling/modules/` is composable:

**Attention mechanisms** (`modules/attention/`):
- `DotProductAttention` — standard scaled dot-product
- `PerceiverAttention` — cross/self attention for perceiver-style models
- `TransolverAttention` — Physics-Attention with learnable slices
- Anchor attention family — `SelfAnchor`, `CrossAnchor`, `Joint`, `MultiBranch`, `Mixed`
- **Registry pattern**: `ATTENTION_REGISTRY` dict for dynamic loading from config strings

**Layers** (`modules/layers/`):
- `ContinuousSincosEmbedding` — continuous positional embeddings for arbitrary coordinates
- `RoPEFrequency` — rotary position embedding frequency computation
- `LayerScale` — per-channel learned scaling (from CaiT)
- `UnquantizedDropPath` — stochastic depth
- `ScalarConditioner` — condition tensors on scalar input (for design parameters)

**Blocks** (`modules/blocks/`):
- `TransformerBlock` — Norm → Attention → Residual → Norm → MLP → Residual
  (supports modulation, layer scale, drop path, configurable attention via registry)
- `PerceiverBlock` — cross-attention from queries to key-value context

**Functional** (`modeling/functional/`):
- `geometric.py` — vector/matrix ops, distance computation (with optional Triton kernels)
- `rope.py` — RoPE application
- `modulation.py` — scale/shift/gate modulation for conditional generation

### Extension points summary

| What to customize | What to subclass | What to override |
|-------------------|-----------------|------------------|
| Training loss | `BaseTrainer` | `loss_compute()` or `train_step()` |
| Model architecture | `Model` or `CompositeModel` | `forward()`, `initialize_weights()` |
| Evaluation metrics | `PeriodicDataIteratorCallback` | `process_data()`, `process_results()` |
| Dataset | `Dataset` | `getitem_*()` methods |
| Data processing | `SampleProcessor` / `BatchProcessor` | `__call__(sample)` |
| LR schedule | `ScheduleBase` | `_get_value(step, total_steps)` |
| Domain preset | `DomainPreset` | `data_specs`, `dataset_statistics`, `normalizer_spec` |
| Attention | Implement attention interface | Register in `ATTENTION_REGISTRY` |

## 6. Hands-on: training via CLI

In [ ]:
import torch

print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()} ({torch.cuda.device_count()} device(s))")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"Memory:   {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

import noether

print(f"Noether:  {noether.__file__}")

In [ ]:
# ── Set data paths for the cluster ──────────────────────────────────
SHAPENET_ROOT = "/path/to/shapenet_car"  # <-- EDIT
AHMEDML_ROOT = "/path/to/ahmedml"  # <-- EDIT
DRIVAERML_ROOT = "/path/to/drivaerml"  # <-- EDIT
OUTPUT_PATH = "./outputs"

The tutorial uses `noether-train` with Hydra YAML configs — the standard approach
for production experiments. All commands run from the repo root.

### Transformer (smoke test — 2 epochs)

In [ ]:
!cd ~/exp/noether && uv run noether-train \
    --hp tutorial/configs/train_shapenet.yaml \
    +experiment/shapenet=transformer \
    tracker=disabled \
    dataset_root={SHAPENET_ROOT} \
    trainer.max_epochs=2 \
    +seed=1

### AB-UPT (smoke test — 2 epochs)

In [ ]:
!cd ~/exp/noether && uv run noether-train \
    --hp tutorial/configs/train_shapenet.yaml \
    +experiment/shapenet=ab_upt \
    tracker=disabled \
    dataset_root={SHAPENET_ROOT} \
    trainer.max_epochs=2 \
    +seed=1

### Transolver

In [ ]:
!cd ~/exp/noether && uv run noether-train \
    --hp tutorial/configs/train_shapenet.yaml \
    +experiment/shapenet=transolver \
    tracker=disabled \
    dataset_root={SHAPENET_ROOT} \
    trainer.max_epochs=2 \
    +seed=1

### Overriding hyperparameters from the CLI

```bash
# Single override
uv run noether-train --hp tutorial/configs/train_shapenet.yaml \
    +experiment/shapenet=transformer trainer.max_epochs=100

# Multiple overrides (hidden_dim % num_heads must == 0)
uv run noether-train --hp tutorial/configs/train_shapenet.yaml \
    +experiment/shapenet=transformer \
    model.hidden_dim=256 model.transformer_block_config.num_heads=4
```

## 7. Hands-on: training via Python preset API

The `examples/aero_cfd/` scripts use the **preset-based API** — a higher-level Python interface
that builds configs programmatically. Each preset encapsulates dataset-specific defaults
(statistics, normalizers, pipeline params, forward property mappings).

| Preset | Dataset | Script |
|--------|---------|--------|
| `ShapeNetCarPreset` | ShapeNet Car | `train_shapenet_car.py` |
| `AhmedMLPreset` | AhmedML (CAEML) | `train_ahmedml.py` |
| `DrivAerMLPreset` | DrivAerML (CAEML) | `train_drivaerml.py` |
| `DrivAerNetPreset` | DrivAerNet++ | `train_drivaernet.py` |
| `EmmiWingPreset` | Emmi Wing | `train_emmi_wing.py` |

Each script provides: `train_abupt`, `train_upt`, `train_transformer`, `train_transolver`.

### ShapeNet-Car — Transformer via preset

In [ ]:
import sys

sys.path.insert(0, "/home/ggalletti/exp/noether")

from examples.aero_cfd import ShapeNetCarPreset

from noether.training.runners import HydraRunner

preset = ShapeNetCarPreset()
config = preset.build_config(
    model_kind="noether.modeling.models.aerodynamics.AeroTransformer",
    model_params=dict(hidden_dim=192, depth=12),
    trainer_kind="noether.training.trainers.WeightedLossTrainer",
    trainer_params=dict(field_weights={"surface_pressure": 1.0, "volume_velocity": 1.0}),
    dataset_root=SHAPENET_ROOT,
    output_path=OUTPUT_PATH,
    max_epochs=2,
    accelerator="cuda",
)
HydraRunner().main(device="cuda", config=config)

### ShapeNet-Car — AB-UPT via preset

In [ ]:
from examples.aero_cfd import ShapeNetCarPreset

from noether.training.runners import HydraRunner

preset = ShapeNetCarPreset()
config = preset.build_config(
    model_kind="noether.modeling.models.aerodynamics.AeroABUPT",
    model_params=dict(
        hidden_dim=192,
        geometry_depth=6,
        physics_blocks=["perceiver"] + ["self", "cross"] * 5,
    ),
    trainer_kind="noether.training.trainers.WeightedLossTrainer",
    trainer_params=dict(field_weights={"surface_pressure": 1.0, "volume_velocity": 1.0}),
    dataset_root=SHAPENET_ROOT,
    output_path=OUTPUT_PATH,
    max_epochs=2,
    accelerator="cuda",
)
HydraRunner().main(device="cuda", config=config)

### AhmedML — AB-UPT via preset

AhmedML has more output fields (5 vs 2 for ShapeNet), so field weights are richer.

In [ ]:
from examples.aero_cfd import AhmedMLPreset

from noether.training.runners import HydraRunner

preset = AhmedMLPreset()
config = preset.build_config(
    model_kind="noether.modeling.models.aerodynamics.AeroABUPT",
    model_params=dict(
        hidden_dim=192,
        geometry_depth=6,
        physics_blocks=["perceiver"] + ["self", "cross"] * 5,
    ),
    trainer_kind="noether.training.trainers.WeightedLossTrainer",
    trainer_params=dict(
        field_weights={
            "surface_pressure": 1.0,
            "surface_friction": 1.0,
            "volume_pressure": 1.0,
            "volume_velocity": 1.0,
            "volume_vorticity": 1.0,
        }
    ),
    dataset_root=AHMEDML_ROOT,
    output_path=OUTPUT_PATH,
    datasets=["train", "val", "test"],
    max_epochs=2,
    accelerator="cuda",
)
HydraRunner().main(device="cuda", config=config)

### DrivAerML — AB-UPT via preset

In [ ]:
from examples.aero_cfd import DrivAerMLPreset

from noether.training.runners import HydraRunner

preset = DrivAerMLPreset()
config = preset.build_config(
    model_kind="noether.modeling.models.aerodynamics.AeroABUPT",
    model_params=dict(
        hidden_dim=192,
        geometry_depth=6,
        physics_blocks=["perceiver"] + ["self", "cross"] * 5,
    ),
    trainer_kind="noether.training.trainers.WeightedLossTrainer",
    trainer_params=dict(
        field_weights={
            "surface_pressure": 1.0,
            "surface_friction": 1.0,
            "volume_pressure": 1.0,
            "volume_velocity": 1.0,
            "volume_vorticity": 1.0,
        }
    ),
    dataset_root=DRIVAERML_ROOT,
    output_path=OUTPUT_PATH,
    datasets=["train", "val", "test"],
    max_epochs=2,
    accelerator="cuda",
)
HydraRunner().main(device="cuda", config=config)

## 8. Hands-on: exploring components

Load individual pieces interactively to inspect datasets, models, and configs.

### Load and inspect the ShapeNet-Car dataset

In [ ]:
from noether.data.datasets.cfd import ShapeNetCarDataset

dataset = ShapeNetCarDataset(
    root=SHAPENET_ROOT,
    split="train",
    excluded_properties=["surface_friction", "volume_pressure", "volume_vorticity"],
)
print(f"Dataset size: {len(dataset)}")

sample = dataset[0]
print("\nSample tensors:")
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(f"  {k:30s} {str(v.shape):20s} {v.dtype}")
    else:
        print(f"  {k:30s} {type(v).__name__}: {v}")

### List available models

In [ ]:
import noether.modeling.models as models_mod

print("Available models:", [x for x in dir(models_mod) if not x.startswith("_")])

### Inspect a preset config

See what the preset API generates — useful for understanding what the YAML configs resolve to.

In [ ]:
from examples.aero_cfd import ShapeNetCarPreset

preset = ShapeNetCarPreset()
config = preset.build_config(
    model_kind="noether.modeling.models.aerodynamics.AeroTransformer",
    model_params=dict(hidden_dim=192, depth=12),
    trainer_kind="noether.training.trainers.WeightedLossTrainer",
    trainer_params=dict(field_weights={"surface_pressure": 1.0, "volume_velocity": 1.0}),
    dataset_root=SHAPENET_ROOT,
    output_path=OUTPUT_PATH,
    max_epochs=2,
    accelerator="cuda",
)
config

## 9. Running at scale on Slurm

For longer runs, submit batch jobs. The tutorial provides job scripts that use
Slurm **job arrays** to sweep over models and seeds.

### View the job script and experiment matrix

In [ ]:
!cat ~/exp/noether/tutorial/jobs/train_shapenet.job

In [ ]:
# 4 models x 5 seeds = 20 runs
!cat ~/exp/noether/tutorial/jobs/experiments/shapenet_experiments.txt

### Submit and monitor

In [ ]:
# Submit the full sweep
# !sbatch ~/exp/noether/tutorial/jobs/train_shapenet.job

# Check job status
!squeue -u $USER

### Multi-GPU training

Outside Slurm, `noether-train` auto-detects GPUs via `CUDA_VISIBLE_DEVICES`.
Inside Slurm, use `srun` with `--ntasks-per-node` matching GPU count:

```bash
srun --nodes=1 --partition=compute --gpus-per-node=2 --mem=64GB \
    --ntasks-per-node=2 --kill-on-bad-exit=1 --cpus-per-task=28 \
    uv run noether-train --hp tutorial/configs/train_shapenet.yaml \
    +experiment/shapenet=transformer tracker=disabled trainer.effective_batch_size=2
```

Set `trainer.effective_batch_size` to the number of GPUs (or higher for gradient accumulation).

## 10. Reference from docs

Additional material sourced from the
[Noether documentation](https://noether-docs.emmi.ai/noether).

### Data pipeline internals

The [`MultiStagePipeline`](https://noether-docs.emmi.ai/noether/understanding_the_data_pipeline.html)
serves as the DataLoader's collation function, coordinating stages 2-4.

**Property discovery:** the framework introspects all `getitem_<property>` methods on the dataset.
By default every property is loaded. Two mechanisms restrict this:
- Config: `excluded_properties` / `included_properties` on dataset config
- Code: wrap with `PropertySubsetWrapper`

**Point sampling:** `PointSamplingSampleProcessor` uses shared random permutations so that
position↔field correspondence is preserved:

```python
PointSamplingSampleProcessor(
    items={"volume_position", "volume_velocity"},
    num_points=4096,
    seed=None,  # stochastic for training, fixed for eval
)
```

**Two-level subsampling** for large meshes (AhmedML, DrivAerML):
1. Offline: raw mesh reduced by `subsample_factor` (default 10x) during preprocessing
2. Online: further subsampled to target count at training time

**Epochs and coverage:** an epoch = one pass over *samples*, not individual points.
With stochastic sampling (`seed=None`), each sample gets different random points every epoch:
- 28,504-point volume sampled to 4,096 → ~14.4% per epoch
- ~87% coverage after 13 epochs, ~95% after 19 epochs

**Interleaved sampling:** rather than separate train/eval DataLoaders, a single
`InterleavedSampler` serves both — yielding training indices while periodically pausing
for complete evaluation passes. Training stops at the first limit reached:
`max_epochs`, `max_updates`, or `max_samples`.

**Performance:** every `__getitem__` reads full tensors via `torch.load` without caching.
Efficiency comes from: property filtering, DataLoader workers (default: `(#CPUs / #GPUs) - 1`),
offline subsampling, and small file sizes (individual `.pt` < 1 MB).

**Dataset statistics** use Welford's online algorithm (`RunningMoments`) in float64 precision:
```bash
noether-dataset-stats \
    --dataset_kind=noether.data.datasets.cfd.ShapeNetCarDataset \
    --split=train --root=/path/to/data \
    --exclude_attributes=surface_friction,volume_pressure,volume_vorticity
```

### Shuffling mechanisms

- **Dataset-level:** `ShuffleWrapper` applies a one-time fixed permutation at construction
  (deterministic subset selection)
- **Sampler-level:** `RandomSampler` (single-GPU) or `DistributedSampler` (multi-GPU) create
  fresh random orderings per epoch. `InterleavedSampler` calls `set_epoch()` to re-seed.

### Config-driven vs code-driven progression

The [docs recommend](https://noether-docs.emmi.ai/noether/key_concepts.html) a progression:

1. **Config-level changes** — modify existing YAML (e.g. adjust model size, switch optimizer)
2. **Basic code customization** — create custom attention blocks / transformer blocks,
   reference them via `kind` in config
3. **Deep code customization** — custom trainers, datasets, pipelines for domain-specific needs

Start with configs and built-in pipelines, then move to deeper customizations when needed.

### Known limitations

From the [design principles page](https://noether-docs.emmi.ai/noether/design_principles_and_limitations.html):

- **Configuration coupling:** many classes accept a single `config` object, making standalone
  use (e.g. in notebooks) difficult without constructing Pydantic objects
- **String-based architecture:** reliance on class path strings means IDE refactoring tools
  may miss references in YAML
- **Factory indirection:** implicit logic in `Factory.instantiate` can obscure object creation
  during debugging
- **Boilerplate:** adding features typically requires modifying implementation, schema, and
  factory files — favours stability over rapid prototyping
- **Nested config complexity:** deeply layered schemas can challenge new users

### Further reading

| Topic | Link |
|-------|------|
| Introduction | [noether-docs.emmi.ai/noether/introduction_to_noether_framework.html](https://noether-docs.emmi.ai/noether/introduction_to_noether_framework.html) |
| Key concepts | [noether-docs.emmi.ai/noether/key_concepts.html](https://noether-docs.emmi.ai/noether/key_concepts.html) |
| Data pipeline deep dive | [noether-docs.emmi.ai/noether/understanding_the_data_pipeline.html](https://noether-docs.emmi.ai/noether/understanding_the_data_pipeline.html) |
| Hardware setup | [noether-docs.emmi.ai/reference/hardware_setup.html](https://noether-docs.emmi.ai/reference/hardware_setup.html) |
| Training with configs | [noether-docs.emmi.ai/tutorials/training_first_model_with_configs.html](https://noether-docs.emmi.ai/tutorials/training_first_model_with_configs.html) |
| Training with code | [noether-docs.emmi.ai/tutorials/training_first_model_with_code.html](https://noether-docs.emmi.ai/tutorials/training_first_model_with_code.html) |
| Custom dataset | [noether-docs.emmi.ai/guides/data/how_to_implement_a_custom_dataset.html](https://noether-docs.emmi.ai/guides/data/how_to_implement_a_custom_dataset.html) |
| Custom model | [noether-docs.emmi.ai/guides/training/implement_a_custom_model.html](https://noether-docs.emmi.ai/guides/training/implement_a_custom_model.html) |
| Custom trainer | [noether-docs.emmi.ai/guides/training/implement_a_custom_trainer.html](https://noether-docs.emmi.ai/guides/training/implement_a_custom_trainer.html) |
| Callbacks | [noether-docs.emmi.ai/guides/training/use_callbacks.html](https://noether-docs.emmi.ai/guides/training/use_callbacks.html) |
| Launch jobs | [noether-docs.emmi.ai/guides/training/launch_job.html](https://noether-docs.emmi.ai/guides/training/launch_job.html) |
| Inference | [noether-docs.emmi.ai/guides/inference/how_to_run_evaluation_on_trained_models.html](https://noether-docs.emmi.ai/guides/inference/how_to_run_evaluation_on_trained_models.html) |
| Config inheritance | [noether-docs.emmi.ai/reference/config_inheritance.html](https://noether-docs.emmi.ai/reference/config_inheritance.html) |
| Full tutorial README | [github.com/Emmi-AI/noether/blob/main/tutorial/README.MD](https://github.com/Emmi-AI/noether/blob/main/tutorial/README.MD) |